# NB04 — Baselines, Smooth Wrapper & Base Controller (Apple Full-Body)

Establish performance baselines for the Apple Full-Body env using Random
and Heuristic policies. Build `SmoothActionWrapper` and `BaseController`
for use in Residual SAC (NB07). Generate a leaderboard table as reference.

**Key outputs:**
- Baseline leaderboard (Random, Stand-Only, Heuristic, Smoothed, BaseController)
- `SmoothActionWrapper` — EMA action filter for jerk reduction
- `BaseController` — heuristic + EMA (used as base in NB07 Residual SAC)


## Objectives

1. Define `evaluate_policy()` helper for consistent evaluation.
2. **Random baseline**: sample random actions, measure reward / fall rate / jerk.
3. **Stand-Only baseline**: zero actions (hold standing pose).
4. **Heuristic — Reach Apple**: proportional control toward apple.
5. **SmoothActionWrapper**: EMA filter to reduce jerk.
6. **Smoothed Random baseline**: demonstrate jerk reduction.
7. **BaseController**: heuristic + EMA (for NB07 Residual SAC).
8. Rank all baselines in a leaderboard table.


## Resources

| Resource | Requirement | Notes |
|----------|-------------|-------|
| GPU | Not required | CPU OK |
| RAM | 8 GB | |
| Runtime | ~10-20 min | 5 baselines × 20 episodes |


## Imports & Setup


In [ ]:
import sys, os, json, random, time
from pathlib import Path

# Force Lavapipe software Vulkan (RunPod has no display GPU driver)
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/lvp_icd.json"
os.environ["MESA_VK_DEVICE_SELECT"] = "10005:0"

import numpy as np
import torch
import gymnasium as gym
import pandas as pd
import matplotlib.pyplot as plt
import mediapy as media
from IPython.display import display, HTML

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

# Robust project root detection
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    if (_cwd / "src" / "envs").is_dir():
        PROJECT_ROOT = _cwd
    elif (_cwd.parent / "src" / "envs").is_dir():
        PROJECT_ROOT = _cwd.parent
    else:
        PROJECT_ROOT = Path("/root/robotic-sim-dishwash")

sys.path.insert(0, str(PROJECT_ROOT))
from src.envs import UnitreeG1PlaceAppleInBowlFullBodyEnv

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB04"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

assert (PROJECT_ROOT / "artifacts" / "NB03" / "reward_contract_apple.json").exists(), \
    "Run NB03 first!"

# ── GPU Check ──
print("=" * 60)
print("  System & GPU Check")
print("=" * 60)
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"  GPU      : {props.name}")
    print(f"  VRAM     : {vram_gb:.1f} GB")
else:
    print("  GPU      : Not available — using CPU")
print(f"  Render   : Lavapipe (software Vulkan)")
print("=" * 60)
print("✅ Prerequisites found")

/root/robotic-sim-dishwash/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/robotic-sim-dishwash/.env/lib/python3.12/site-packages/sapien/__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


  System & GPU Check
  PyTorch  : 2.10.0+cu128
  CUDA     : True
  GPU      : NVIDIA GeForce RTX 5090


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## Configuration


In [ ]:
CFG = {
    "seed": 42,
    "env_id": "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "control_mode": "pd_joint_delta_pos",
    "obs_mode": "state",
    "n_eval_episodes": 20,
    "max_steps_per_ep": 200,
    "smooth_alpha": 0.3,
}

SEED = CFG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Config loaded")


## Step 1 — `evaluate_policy()` Helper


In [ ]:
def evaluate_policy(env, policy_fn, n_episodes, max_steps, seed=42):
    """Run policy and collect per-episode metrics.

    Args:
        policy_fn: callable(obs, info) → action
    Returns:
        list of dicts with per-episode metrics
    """
    results = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        ep_reward, ep_steps = 0.0, 0
        prev_action = np.zeros(env.action_space.shape)
        jerks = []
        fell, success = False, False

        for step in range(max_steps):
            action = policy_fn(obs, info)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += float(reward)
            ep_steps += 1

            jerk = float(np.sum((action - prev_action) ** 2))
            jerks.append(jerk)
            prev_action = action.copy()

            if terminated:
                success = bool(info.get("success", False))
                fell = not success
                break
            if truncated:
                break

        results.append({
            "episode": ep,
            "total_reward": ep_reward,
            "steps": ep_steps,
            "success": success,
            "fell": fell,
            "mean_jerk": float(np.mean(jerks)) if jerks else 0.0,
        })
    return results


# ── Video helpers (same pattern as ref-code) ──

def to_uint8(frame):
    """Convert render output to uint8 numpy array."""
    if isinstance(frame, torch.Tensor):
        frame = frame.cpu().numpy()
    if frame.ndim == 4:
        frame = frame[0]
    if frame.dtype in (np.float32, np.float64):
        frame = (frame * 255).clip(0, 255).astype(np.uint8)
    return frame


def collect_episode_video(env, policy_fn, max_steps=200, seed=42):
    """Collect frames + rewards for one episode — ref-code style."""
    frames, rewards = [], []
    obs, info = env.reset(seed=seed)
    frames.append(to_uint8(env.render()))
    for step in range(max_steps):
        action = policy_fn(obs, info)
        obs, reward, terminated, truncated, info = env.step(action)
        frames.append(to_uint8(env.render()))
        rewards.append(float(reward))
        if terminated or truncated:
            break
    return frames, rewards

print("✅ evaluate_policy() + video helpers defined")

## Step 2 — Random Baseline


In [ ]:
env = gym.make(
    CFG["env_id"], num_envs=1, obs_mode=CFG["obs_mode"],
    control_mode=CFG["control_mode"], render_mode="rgb_array",
    render_backend="cpu",
)
env = CPUGymWrapper(env)

def random_policy(obs, info):
    return env.action_space.sample()

random_results = evaluate_policy(
    env, random_policy, CFG["n_eval_episodes"], CFG["max_steps_per_ep"], SEED
)

rews = [r["total_reward"] for r in random_results]
print(f"Random: mean_reward={np.mean(rews):.4f} ± {np.std(rews):.4f}")
print(f"  fall_rate={np.mean([r['fell'] for r in random_results]):.2%}")
print(f"  mean_jerk={np.mean([r['mean_jerk'] for r in random_results]):.4f}")

## Step 3 — Stand-Only Baseline

Zero actions = hold current joint positions (delta=0).
Demonstrates survival without manipulation.


In [ ]:
def stand_only_policy(obs, info):
    """Zero delta on all joints — hold standing pose."""
    return np.zeros(env.action_space.shape, dtype=np.float32)

stand_results = evaluate_policy(
    env, stand_only_policy, CFG["n_eval_episodes"], CFG["max_steps_per_ep"], SEED
)

rews = [r["total_reward"] for r in stand_results]
print(f"Stand-Only: mean_reward={np.mean(rews):.4f} ± {np.std(rews):.4f}")
print(f"  fall_rate={np.mean([r['fell'] for r in stand_results]):.2%}")


## Step 4 — Heuristic: Reach Apple

Proportional control: move hand toward apple position.
Arm joints get directional deltas, leg joints stay at zero.


In [ ]:
def heuristic_reach_policy(obs, info):
    """Simple proportional control: move arm joints toward apple direction.

    Strategy:
    - Keep legs at zero delta (hold standing pose)
    - Move arm/hand joints with small proportional signals
    - This is a rough heuristic — exact joint-to-Cartesian mapping is complex
    """
    action = np.zeros(env.action_space.shape, dtype=np.float32)

    # We use small random arm signals biased toward reaching
    # In a real heuristic, we'd compute IK or use Cartesian error
    # For now, apply small positive delta to arm joints to extend arm forward
    act_dim = env.action_space.shape[0]

    # Approximate arm joint indices (after legs and torso)
    # Legs: ~12 joints, Torso: ~1 joint, Arms: ~10 joints, Hands: ~14 joints
    arm_start = 13  # approximate
    arm_end = 23    # approximate

    # Small reaching signal on arm joints
    action[arm_start:arm_end] = np.random.uniform(-0.1, 0.1, arm_end - arm_start)
    action[arm_start:arm_end] *= 0.5  # conservative

    return action.astype(np.float32)

reach_results = evaluate_policy(
    env, heuristic_reach_policy, CFG["n_eval_episodes"], CFG["max_steps_per_ep"], SEED
)

rews = [r["total_reward"] for r in reach_results]
print(f"Heuristic Reach: mean_reward={np.mean(rews):.4f} ± {np.std(rews):.4f}")
print(f"  fall_rate={np.mean([r['fell'] for r in reach_results]):.2%}")


## Step 5 — SmoothActionWrapper

EMA (Exponential Moving Average) action filter:
$$a_{smooth} = \alpha \cdot a_{raw} + (1 - \alpha) \cdot a_{prev}$$

Where $\alpha = 0.3$ means 30% new action, 70% previous.
This dramatically reduces jerk (action oscillation).


In [ ]:
class SmoothActionWrapper(gym.ActionWrapper):
    """EMA action filter for jerk reduction."""

    def __init__(self, env, alpha=0.3):
        super().__init__(env)
        self.alpha = alpha
        self._prev_action = None

    def action(self, action):
        if self._prev_action is None:
            self._prev_action = action.copy()
        smoothed = self.alpha * action + (1 - self.alpha) * self._prev_action
        self._prev_action = smoothed.copy()
        return smoothed

    def reset(self, **kwargs):
        self._prev_action = None
        return self.env.reset(**kwargs)

print(f"✅ SmoothActionWrapper defined (alpha={CFG['smooth_alpha']})")


## Step 6 — Smoothed Random Baseline


In [ ]:
env_smooth = SmoothActionWrapper(env, alpha=CFG["smooth_alpha"])

smooth_random_results = evaluate_policy(
    env_smooth, random_policy, CFG["n_eval_episodes"],
    CFG["max_steps_per_ep"], SEED,
)

rews_s = [r["total_reward"] for r in smooth_random_results]
jerks_s = [r["mean_jerk"] for r in smooth_random_results]
jerks_raw = [r["mean_jerk"] for r in random_results]
print(f"Smoothed Random: mean_reward={np.mean(rews_s):.4f}")
print(f"  Jerk reduction: {np.mean(jerks_raw):.4f} → {np.mean(jerks_s):.4f} "
      f"({(1 - np.mean(jerks_s)/np.mean(jerks_raw))*100:.0f}% reduction)")


## Step 7 — BaseController (for Residual SAC in NB07)

Combines heuristic policy with EMA smoothing.
In NB07: `a_final = clip(a_base + β × a_residual)`


In [ ]:
class BaseController:
    """Heuristic + EMA smoothing. Produces base actions for residual learning.

    Usage in NB07 Residual SAC:
        base_ctrl = BaseController(env, alpha=0.3)
        a_base = base_ctrl.get_action(obs)
        a_final = np.clip(a_base + beta * a_residual, low, high)
    """

    def __init__(self, env, alpha=0.3):
        self.env = env
        self.alpha = alpha
        self._prev_action = None

    def get_action(self, obs, info=None):
        raw = heuristic_reach_policy(obs, info or {})
        if self._prev_action is None:
            self._prev_action = raw.copy()
        smoothed = self.alpha * raw + (1 - self.alpha) * self._prev_action
        self._prev_action = smoothed.copy()
        return smoothed

    def reset(self):
        self._prev_action = None

# Test
base_ctrl = BaseController(env, alpha=CFG["smooth_alpha"])

def base_ctrl_policy(obs, info):
    return base_ctrl.get_action(obs, info)

base_ctrl_results = evaluate_policy(
    env, base_ctrl_policy, CFG["n_eval_episodes"], CFG["max_steps_per_ep"], SEED
)

rews_bc = [r["total_reward"] for r in base_ctrl_results]
print(f"BaseController: mean_reward={np.mean(rews_bc):.4f} ± {np.std(rews_bc):.4f}")
print(f"  fall_rate={np.mean([r['fell'] for r in base_ctrl_results]):.2%}")
print(f"  mean_jerk={np.mean([r['mean_jerk'] for r in base_ctrl_results]):.4f}")


## Step 8 — Baseline Leaderboard


In [ ]:
def summarize(results, name):
    rewards = [r["total_reward"] for r in results]
    return {
        "method":       name,
        "mean_reward":  float(np.mean(rewards)),
        "std_reward":   float(np.std(rewards)),
        "success_rate": float(np.mean([r["success"] for r in results])),
        "fall_rate":    float(np.mean([r["fell"] for r in results])),
        "mean_jerk":    float(np.mean([r["mean_jerk"] for r in results])),
        "mean_steps":   float(np.mean([r["steps"] for r in results])),
    }

leaderboard = pd.DataFrame([
    summarize(random_results,        "Random"),
    summarize(stand_results,         "Stand-Only"),
    summarize(reach_results,         "Heuristic (Reach)"),
    summarize(smooth_random_results, "Smoothed Random"),
    summarize(base_ctrl_results,     "BaseController"),
])
leaderboard = leaderboard.sort_values("mean_reward", ascending=False)
leaderboard.to_csv(ARTIFACTS_DIR / "baseline_leaderboard.csv", index=False)

print("\nBaseline Leaderboard:")
print(leaderboard.to_string(index=False))


## Step 9 — Baseline Policy Videos (ref-code style)

Record short episodes for each baseline and display inline.
This shows how the robot behaves under each policy.

In [ ]:
# Record short video episodes for each baseline policy
VIDEO_STEPS = 100  # shorter for video display
VIDEO_FPS   = 20
VIDEO_DIR   = ARTIFACTS_DIR / "videos"
VIDEO_DIR.mkdir(exist_ok=True)

base_ctrl.reset()  # reset BaseController state
policies = {
    "01_random":       random_policy,
    "02_stand_only":   stand_only_policy,
    "03_heuristic":    heuristic_reach_policy,
    "04_base_ctrl":    base_ctrl_policy,
}

video_data = {}
for name, pol_fn in policies.items():
    if name == "04_base_ctrl":
        base_ctrl.reset()
    print(f"Recording {name} …", end=" ", flush=True)
    frames, rews = collect_episode_video(env, pol_fn, max_steps=VIDEO_STEPS, seed=SEED)
    video_data[name] = {"frames": frames, "rewards": rews}

    # Save .mp4 to disk
    mp4_path = VIDEO_DIR / f"{name}.mp4"
    media.write_video(str(mp4_path), frames, fps=VIDEO_FPS)
    print(f"✅ {len(frames)} frames, reward={sum(rews):.3f}")

print(f"\nVideos saved to: {VIDEO_DIR}")

In [ ]:
# ── Show one representative video inline (Stand-Only baseline) ──
# Stand-only is the simplest — robot just holds pose, good reference
print("▶  Stand-Only policy video (100 steps):")
media.show_video(video_data["02_stand_only"]["frames"], fps=VIDEO_FPS)

In [ ]:
# ── Per-step reward curves for each baseline (from video episodes) ──
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
colors_vid = {"01_random": "gray", "02_stand_only": "gold",
              "03_heuristic": "dodgerblue", "04_base_ctrl": "seagreen"}
titles_vid = {"01_random": "Random", "02_stand_only": "Stand-Only",
              "03_heuristic": "Heuristic (Reach)", "04_base_ctrl": "BaseController"}

for ax, (name, data) in zip(axes.flat, video_data.items()):
    rews = data["rewards"]
    cum_rew = np.cumsum(rews)
    ax.plot(rews, alpha=0.6, color=colors_vid[name], lw=0.8, label="per-step")
    ax2 = ax.twinx()
    ax2.plot(cum_rew, color=colors_vid[name], lw=2, ls="--", label="cumulative")
    ax.set_title(f"{titles_vid[name]}  (total={sum(rews):.2f})", fontsize=11)
    ax.set_ylabel("Step Reward")
    ax2.set_ylabel("Cumulative")
    ax.grid(True, alpha=0.2)
    ax.axhline(0, color="black", lw=0.5)

axes[1, 0].set_xlabel("Step")
axes[1, 1].set_xlabel("Step")
fig.suptitle("Per-Step Reward Curves — Baseline Policies (Apple Full-Body)",
             fontweight="bold", fontsize=13)
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "reward_curves_per_step.png", dpi=150)
plt.show()
print("Saved: reward_curves_per_step.png")

In [ ]:
# ── Show Random policy video for contrast ──
print("▶  Random policy video (100 steps):")
media.show_video(video_data["01_random"]["frames"], fps=VIDEO_FPS)

## Step 10 — Comparison Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

methods = leaderboard["method"].tolist()
colors = ["gray", "khaki", "dodgerblue", "lightcoral", "mediumseagreen"][:len(methods)]

# Reward
axes[0].bar(methods, leaderboard["mean_reward"].tolist(),
            yerr=leaderboard["std_reward"].tolist(),
            color=colors, edgecolor="black", capsize=3)
axes[0].set_title("Mean Reward by Baseline")
axes[0].set_ylabel("Mean Total Reward")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=20, ha="right")
axes[0].grid(axis="y", alpha=0.3)

# Jerk
axes[1].bar(methods, leaderboard["mean_jerk"].tolist(),
            color=colors, edgecolor="black")
axes[1].set_title("Mean Jerk (lower = smoother)")
axes[1].set_ylabel("Mean Jerk")
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha="right")
axes[1].grid(axis="y", alpha=0.3)

# Fall rate
axes[2].bar(methods, leaderboard["fall_rate"].tolist(),
            color=colors, edgecolor="black")
axes[2].set_title("Fall Rate (lower = better)")
axes[2].set_ylabel("Fall Rate")
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=20, ha="right")
axes[2].grid(axis="y", alpha=0.3)

fig.suptitle("NB04 — Baseline Comparison (Apple Full-Body)", fontweight="bold")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "baseline_comparison.png", dpi=150)
plt.show()


## Save Artifacts


In [ ]:
with open(ARTIFACTS_DIR / "nb04_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

env.close()
print("✅ NB04 Baselines & Smoothing Complete")
print(f"Artifacts saved to: {ARTIFACTS_DIR}")
print(f"Videos saved to: {VIDEO_DIR}")
print(f"  - 01_random.mp4, 02_stand_only.mp4, 03_heuristic.mp4, 04_base_ctrl.mp4")

## Artifacts

| File | Description |
|------|-------------|
| `baseline_leaderboard.csv` | All baselines ranked by mean reward |
| `baseline_comparison.png` | Bar charts comparing baselines |
| `nb04_config.json` | Config used |

## Key Definitions for Downstream Use

- **`SmoothActionWrapper`**: `a_smooth = α·a_raw + (1-α)·a_prev` — used in NB07
- **`BaseController`**: heuristic + EMA — used in NB07 Residual SAC
- **`evaluate_policy()`**: consistent eval helper — pattern reused in NB05-NB09
